# Lab 02 — Feedforward networks and backpropagation

**Dataset:** [ylecun/mnist](https://huggingface.co/datasets/ylecun/mnist), binary 0-vs-1 classification. We will compare a shallow linear baseline with a nonlinear MLP, verify one gradient numerically, use validation data for comparisons, and reserve the test split for one final evaluation.


> **Experiment contract:** do not change the intervention before writing your prediction. Every run reports the same metric and budget so the comparison is interpretable. The test split is reserved for a final check after decisions are made.


## The math behind the experiment

A two-layer network computes a hidden activation, then an output **logit** (a score before converting it to a probability):

$$
z^{(1)}=W^{(1)}x+b^{(1)},\qquad h=\sigma(z^{(1)}),\qquad z^{(2)}=W^{(2)}h+b^{(2)}.
$$

For an interpretable probability, apply the sigmoid: $p=\sigma(z^{(2)})$. In the code, however, `BCEWithLogitsLoss` receives the logit $z^{(2)}$, not the already-sigmoided probability. It combines sigmoid and binary cross-entropy in a numerically stable operation. The model classes therefore return logits; sigmoid is used only when converting them to probabilities for accuracy.

Backpropagation applies the chain rule. For one scalar path, the idea is

$$
\frac{dL}{dw}=\frac{dL}{d\hat y}\frac{d\hat y}{dz_2}\frac{dz_2}{dh}\frac{dh}{dz_1}\frac{dz_1}{dw}.
$$

In a real multi-unit network these local derivatives are Jacobians and are combined with vector/matrix products. Automatic differentiation computes the required products efficiently. Gradient descent then updates a parameter using

$$
w\leftarrow w-\eta\frac{\partial L}{\partial w}.
$$

The finite-difference check below independently approximates one derivative. Agreement with autograd is a useful implementation test; it does not by itself prove that the model, data split, or experiment is correct.

In [ ]:
!pip -q install datasets scikit-learn torch pandas matplotlib
from datasets import load_dataset
import numpy as np, pandas as pd, torch
from sklearn.metrics import accuracy_score
seed=11; torch.manual_seed(seed)
tr=load_dataset("ylecun/mnist",split="train[:4000]"); te=load_dataset("ylecun/mnist",split="test[:1000]")
def xy(ds):
 X=np.array([np.asarray(x).reshape(-1) for x in ds["image"]],np.float32)/255.; y=np.array(ds["label"]); m=(y<2); return torch.tensor(X[m]),torch.tensor(y[m],dtype=torch.float32)
Xall,yall=xy(tr); Xtest,ytest=xy(te)
n_val=max(1,int(0.2*len(Xall)))
split_rng=np.random.default_rng(seed)
indices=split_rng.permutation(len(Xall))
val_idx,train_idx=indices[:n_val],indices[n_val:]
Xtr,ytr=Xall[train_idx],yall[train_idx]
Xval,yval=Xall[val_idx],yall[val_idx]
print({"train":tuple(Xtr.shape),"validation":tuple(Xval.shape),"test":tuple(Xtest.shape)})

## Step 1 — Prediction

Predict that a nonlinear hidden layer will improve over a linear classifier because it can transform the input into a representation that makes the decision easier. Predict that shuffled labels will destroy held-out accuracy.

In [ ]:
import torch.nn as nn
class Linear(nn.Module):
 def __init__(self): super().__init__(); self.net=nn.Linear(784,1)
 def forward(self,x): return self.net(x).squeeze(-1)
class MLP(nn.Module):
 def __init__(self,width=64): super().__init__(); self.net=nn.Sequential(nn.Linear(784,width),nn.ReLU(),nn.Linear(width,1))
 def forward(self,x): return self.net(x).squeeze(-1)
def train(model,epochs=8,lr=1e-2):
 opt=torch.optim.SGD(model.parameters(),lr=lr); lossfn=nn.BCEWithLogitsLoss()
 for _ in range(epochs):
  opt.zero_grad(); loss=lossfn(model(Xtr),ytr); loss.backward(); opt.step()
 with torch.no_grad():
  a=((torch.sigmoid(model(Xtr))>.5)==ytr.bool()).float().mean().item(); b=((torch.sigmoid(model(Xval))>.5)==yval.bool()).float().mean().item()
 return a,b
rows=[]
for name,m in [("linear",Linear()),("MLP-64",MLP(64)),("MLP-128",MLP(128))]:
 a,b=train(m); rows.append([name,a,b])
pd.DataFrame(rows,columns=["model","train_accuracy","validation_accuracy"])

## Step 2 — Numerical gradient check

Autograd should agree with a finite-difference approximation. This is a debugging experiment: if the two disagree materially, do not trust later training results.

In [ ]:
torch.manual_seed(3)
w=torch.tensor([0.7],dtype=torch.double,requires_grad=True); x=torch.tensor([2.0],dtype=torch.double); y=torch.tensor([1.0],dtype=torch.double)
loss=((w*x-y)**2).mean(); loss.backward(); analytic=w.grad.item()
eps=1e-6
with torch.no_grad():
 lp=(( (w+eps)*x-y)**2).mean().item(); lm=(( (w-eps)*x-y)**2).mean().item()
numeric=(lp-lm)/(2*eps)
print({"analytic_gradient":analytic,"numeric_gradient":numeric,"absolute_error":abs(analytic-numeric)})

## Step 3 — Width intervention and shuffled-label control

Keep optimizer and epochs fixed. Change only hidden width. Then permute training labels and repeat the same MLP. Compare train vs validation behavior.

In [ ]:
rng=np.random.default_rng(seed); ysh=torch.tensor(rng.permutation(ytr.numpy()),dtype=torch.float32)
def train_labels(width,labels):
 m=MLP(width); opt=torch.optim.SGD(m.parameters(),lr=1e-2); lf=nn.BCEWithLogitsLoss()
 for _ in range(8): opt.zero_grad(); lf(m(Xtr),labels).backward(); opt.step()
 with torch.no_grad(): return ((torch.sigmoid(m(Xtr))>.5)==labels.bool()).float().mean().item(),((torch.sigmoid(m(Xval))>.5)==yval.bool()).float().mean().item()
rows=[]
for w in [16,64,128]:
 a,b=train_labels(w,ytr); rows.append(["real",w,a,b]); a,b=train_labels(w,ysh); rows.append(["shuffled",w,a,b])
pd.DataFrame(rows,columns=["labels","width","train_accuracy","validation_accuracy"])

## Read the graphs — capacity and generalization

The left plot compares training and validation accuracy for the real-label and shuffled-label runs. High training accuracy alone is not evidence of learning: a model may memorize arbitrary labels. The right plot isolates the numerical gradient check, comparing the derivative from backpropagation with a finite-difference estimate.

For the gradient check, the two bars should be almost indistinguishable. The printed absolute error is the more precise diagnostic.

In [ ]:
import matplotlib.pyplot as plt

width_results = pd.DataFrame(rows, columns=["labels", "width", "train_accuracy", "validation_accuracy"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for label_name, group in width_results.groupby("labels"):
    axes[0].plot(group["width"], group["validation_accuracy"], marker="o", label=f"{label_name}: validation")
    axes[0].plot(group["width"], group["train_accuracy"], marker="x", linestyle="--", label=f"{label_name}: train")
axes[0].set_title("Capacity and generalization")
axes[0].set_xlabel("Hidden-layer width")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0, 1)
axes[0].legend(fontsize=8)
axes[0].grid(alpha=.25)

axes[1].bar(["Backprop", "Finite difference"], [analytic, numeric])
axes[1].set_title("Gradient check")
axes[1].set_ylabel("Estimated derivative")
axes[1].ticklabel_format(axis="y", style="plain")
axes[1].grid(axis="y", alpha=.25)
fig.tight_layout()
plt.show()

## Step 4 — Optimizer and regularization interventions

We now hold the MLP architecture and train/validation split fixed.

1. Compare SGD and Adam from identical initial weights, for the same number of full-batch epochs.
2. Separately compare Adam with weight decay disabled versus enabled, again from identical initial weights.

These are controlled *small teaching experiments*, not a universal optimizer leaderboard. The learning rates are stated explicitly; each optimizer may need its own validation-based tuning. Never choose settings based on the final test set.

In [ ]:
from copy import deepcopy

torch.manual_seed(222)
template=MLP(64)
initial_state=deepcopy(template.state_dict())
loss_fn=nn.BCEWithLogitsLoss()

def train_dynamics(optimizer_name,lr,weight_decay=0.0,epochs=20):
    model=MLP(64)
    model.load_state_dict(deepcopy(initial_state))
    if optimizer_name=="SGD":
        optimizer=torch.optim.SGD(model.parameters(),lr=lr,weight_decay=weight_decay)
    elif optimizer_name=="Adam":
        optimizer=torch.optim.Adam(model.parameters(),lr=lr,weight_decay=weight_decay)
    else:
        raise ValueError(optimizer_name)
    history=[]
    for epoch in range(1,epochs+1):
        model.train()
        optimizer.zero_grad()
        train_loss=loss_fn(model(Xtr),ytr)
        train_loss.backward()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            tr_logits=model(Xtr); te_logits=model(Xval)
            history.append({
                "condition":f"{optimizer_name}, lr={lr:g}, wd={weight_decay:g}",
                "epoch":epoch,
                "train_loss":float(loss_fn(tr_logits,ytr).item()),
                "validation_loss":float(loss_fn(te_logits,yval).item()),
                "train_accuracy":float(((torch.sigmoid(tr_logits)>.5)==ytr.bool()).float().mean().item()),
                "validation_accuracy":float(((torch.sigmoid(te_logits)>.5)==yval.bool()).float().mean().item()),
                "parameters":sum(p.numel() for p in model.parameters())
            })
    return history

dynamics_rows=[]
# Same initialization, architecture, split, full-batch order, and epoch budget.
dynamics_rows += train_dynamics("SGD",lr=0.05)
dynamics_rows += train_dynamics("Adam",lr=0.001)
# Isolate weight decay with Adam fixed.
dynamics_rows += train_dynamics("Adam",lr=0.001,weight_decay=1e-3)
dynamics=pd.DataFrame(dynamics_rows)
assert dynamics["parameters"].nunique()==1
display(dynamics[dynamics["epoch"].isin([1,5,10,20])].pivot(index="epoch",columns="condition",values="validation_accuracy"))

In [ ]:
import matplotlib.pyplot as plt

fig,axes=plt.subplots(1,2,figsize=(13,4.8))
for condition,group in dynamics.groupby("condition"):
    axes[0].plot(group["epoch"],group["train_loss"],label=condition)
    axes[1].plot(group["epoch"],group["validation_accuracy"],label=condition)
axes[0].set_title("Training loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("BCE loss")
axes[1].set_title("Validation accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0,1)
for ax in axes:
    ax.grid(alpha=.25)
    ax.legend(fontsize=7)
fig.tight_layout()
plt.show()

### Answer key: interpret the curves, not just the winner

The architecture, parameter count, initialization, training/validation split, and epoch budget are fixed. SGD versus Adam changes the update rule; Adam with versus without weight decay changes the regularization intervention. Look for both optimization speed and the gap between training and validation behavior.

A condition that wins at epoch 20 is not automatically best after hyperparameter tuning. These are single-seed demonstrations; repeat across seeds and tune hyperparameters using the validation split before making a research claim. The final test split remains untouched during tuning.

## Conclusion / answer key

Expected: the nonlinear MLP should generally outperform the linear model; numerical and analytic gradients should agree closely; shuffled labels can produce high training accuracy with poor validation accuracy when capacity is sufficient.

**Interpretation rule:** do not say “deeper is better” unless the parameter budget and training budget make that comparison fair.

### Research extension
Compare depth and width at approximately equal parameter count, then report mean ± standard deviation over three seeds.

## Final check on the untouched test set

All model and optimizer comparisons above use the validation split. For a clean final estimate, we now train one **predeclared** configuration (MLP with width 64, Adam at learning rate 0.001, no weight decay) on the combined training-plus-validation data after the configuration is fixed, then evaluate it on the test split once. Do not use this test result to choose another model or tune hyperparameters.


In [ ]:
final_model=MLP(64)
final_optimizer=torch.optim.Adam(final_model.parameters(),lr=1e-3)
final_loss=nn.BCEWithLogitsLoss()
for _ in range(20):
    final_optimizer.zero_grad()
    final_loss(final_model(Xall),yall).backward()
    final_optimizer.step()
final_model.eval()
with torch.no_grad():
    test_logits=final_model(Xtest)
    test_bce=float(final_loss(test_logits,ytest).item())
    test_acc=float(((torch.sigmoid(test_logits)>.5)==ytest.bool()).float().mean().item())
print({"predeclared_model":"MLP-64 + Adam(lr=0.001), 20 full-batch epochs", "test_bce":test_bce,"test_accuracy":test_acc})
